# Notebook 05: Model Training
## Forecasting Education Performance -- Tanzania BEST Datasets (2020-2024)
**Author:** Habil Masawika | **Project:** Tanzania BEST ML Forecasting

---

### Objectives
1. Prepare the modelling dataset with a temporally sound train/test split
2. Train 12 regression models from baseline dummies to ensemble methods
3. Evaluate all models using Leave-One-Year-Out cross-validation
4. Compare models on MAE, RMSE, and R² in a ranked comparison table
5. Save all trained model artefacts for evaluation and interpretation notebooks

### Modelling Design
The target variable (CSEE pass rate) is national-level -- identical for all 26 regions in a
given year. The panel regression exploits regional cross-sectional variation in inputs as
explanatory signal alongside national temporal variation.

**Train/Test Split:** 2020-2023 (train) / 2024 (test) -- temporal hold-out.

**Cross-Validation:** Leave-One-Year-Out (LOGO) -- each fold withholds one complete year,
preventing any form of temporal data leakage during model selection.

In [ ]:
import sys, warnings
sys.path.insert(0, '/home/claude/BEST-ML-Forecasting/src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utilities import setup_logging, set_seeds, ProjectPaths, save_dataframe, save_model, Timer
from feature_engineering import get_model_features
from models import ModelEvaluator, get_model_registry, compute_metrics
from preprocessing import scale_features

set_seeds(42)
logger = setup_logging()
paths  = ProjectPaths()

panel_fe = pd.read_csv(paths.processed('best_panel_features.csv'))
print(f"Loaded engineered panel: {panel_fe.shape}")

In [ ]:
# ── Prepare modelling dataset ─────────────────────────────────────────────
TARGET = 'csee_pass_rate'
model_feats = get_model_features(panel_fe, target=TARGET,
                                  include_lags=True, include_interactions=True)

model_df = panel_fe[['year', 'region'] + model_feats + [TARGET]].dropna(
    subset=model_feats + [TARGET]).copy()

print(f"Modelling dataset: {model_df.shape[0]} rows")
print(f"Features: {len(model_feats)}")
print(f"Feature list: {model_feats}")
print(f"\nTarget stats:")
print(model_df[TARGET].describe().round(3))

In [ ]:
# ── Train/test split ──────────────────────────────────────────────────────
TRAIN_YEARS = [2020, 2021, 2022, 2023]
TEST_YEAR   = [2024]

train = model_df[model_df['year'].isin(TRAIN_YEARS)]
test  = model_df[model_df['year'].isin(TEST_YEAR)]

X_train_raw = train[model_feats].values
y_train     = train[TARGET].values
X_test_raw  = test[model_feats].values
y_test      = test[TARGET].values

X_train_sc, X_test_sc, scaler = scale_features(
    pd.DataFrame(X_train_raw, columns=model_feats),
    pd.DataFrame(X_test_raw,  columns=model_feats),
    method='standard'
)

groups = train['year'].values

print(f"Training set: {X_train_sc.shape}  (years {TRAIN_YEARS})")
print(f"Test set:     {X_test_sc.shape}   (year  {TEST_YEAR})")
print(f"y_train range: {y_train.min():.2f} -- {y_train.max():.2f}")

In [ ]:
# ── Train and compare all models ──────────────────────────────────────────
models = get_model_registry()
evaluator = ModelEvaluator(X_train_sc, y_train, groups=groups)

with Timer("Model training and CV"):
    cv_results = evaluator.evaluate_all(models, cv_strategy='logo')

print("\nLeave-One-Year-Out Cross-Validation Results (ranked by CV MAE):")
print(cv_results[['Model','CV_MAE_mean','CV_MAE_std','CV_R2_mean','CV_R2_std','Fit_Time_s']].to_string(index=True))

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────
test_results = evaluator.evaluate_test(X_test_sc, y_test, n_features=len(model_feats))
print("\nTest Set Evaluation (2024 hold-out, ranked by MAE):")
print(test_results[['Model','R2','MAE','RMSE','MedAE','MAPE_pct']].to_string(index=True))

In [ ]:
# ── Visualise model comparison ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (metric, label, asc) in zip(axes, [
    ('CV_MAE_mean',  'CV MAE (%)',  True),
    ('CV_R2_mean',   'CV R²',       False),
    ('CV_MAE_std',   'CV MAE Std',  True),
]):
    df_plot = cv_results.sort_values(metric, ascending=asc)
    colours = ['crimson' if i == 0 else 'steelblue' for i in range(len(df_plot))]
    bars = ax.barh(df_plot['Model'], df_plot[metric], color=colours, alpha=0.85)
    bars[0].set_edgecolor('black'); bars[0].set_linewidth(2)
    ax.set_xlabel(label)
    ax.set_title(f'Model Comparison -- {label}', fontweight='bold')
    ax.invert_yaxis()

plt.suptitle('Model Comparison: Leave-One-Year-Out Cross-Validation', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(paths.fig('nb05_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save comparison tables ────────────────────────────────────────────────
save_dataframe(cv_results,   paths.table('cv_model_comparison.csv'))
save_dataframe(test_results, paths.table('test_model_results.csv'))

# Save all trained models
for name, model in evaluator.trained_models.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    save_model(model, paths.model(f'{safe_name}.pkl'))

save_model(scaler, paths.model('feature_scaler.pkl'))
save_dataframe(pd.DataFrame({'feature': model_feats}),
               paths.table('model_features.csv'))

print("Models and artefacts saved.")
print(f"Best model by CV MAE: {cv_results.iloc[0]['Model']}")